# Comprensión y EDA

# Comprensión de datos y Análisis Exploratorio (EDA)


Librerias que se van a utilizar

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

In [17]:
df = pd.read_csv("../Base_de_datos.csv", parse_dates=["fecha_prestamo"])
df.shape

(10763, 23)

In [18]:
resumen = pd.DataFrame({
    "tipo_dato": df.dtypes,
    "valores_unicos": df.nunique(),
    "ejemplo": df.iloc[0]
})
resumen

,tipo_dato,valores_unicos,ejemplo
tipo_credito,int64,6,4
fecha_prestamo,datetime64[us],10758,2025-01-07 14:40:36
capital_prestado,float64,7306,1852560.0
plazo_meses,int64,18,12
edad_cliente,int64,54,32
tipo_laboral,str,2,Empleado
salario_cliente,int64,1385,3500000
total_otros_prestamos,int64,1538,1000000
cuota_pactada,int64,9836,128650
puntaje,float64,248,95.227787


### Caracterización de variables

| Variable | Clasificación | Observación |
|---|---|---|
| `tipo_credito` | Categórica nominal politómica | Es un código (4, 9, 10, 6, 7, 68), no una cantidad |
| `fecha_prestamo` | Fecha | Se convirtió a tipo fecha al cargar |
| `capital_prestado` | Numérica continua | |
| `plazo_meses` | Numérica discreta | |
| `edad_cliente` | Numérica discreta | |
| `tipo_laboral` | Categórica nominal dicotómica | Empleado / Independiente |
| `salario_cliente` | Numérica continua | |
| `total_otros_prestamos` | Numérica continua | |
| `cuota_pactada` | Numérica continua | |
| `puntaje` | Numérica continua | Puntaje interno |
| `puntaje_datacredito` | Numérica discreta | Puntaje de buró de crédito |
| `cant_creditosvigentes` | Numérica discreta | |
| `huella_consulta` | Numérica discreta | Número de consultas en centrales de riesgo |
| `saldo_mora` | Numérica continua | |
| `saldo_total` | Numérica continua | |
| `saldo_principal` | Numérica continua | |
| `saldo_mora_codeudor` | Numérica continua | Solo 4 valores distintos|
| `creditos_sectorFinanciero` | Numérica discreta | |
| `creditos_sectorCooperativo` | Numérica discreta | |
| `creditos_sectorReal` | Numérica discreta | |
| `promedio_ingresos_datacredito` | Numérica continua | |
| `tendencia_ingresos` | Categórica ordinal politómica | Decreciente < Estable < Creciente →  Es Ordinal 
| `Pago_atiempo` | Categórica dicotómica | **Variable objetivo** (1 = pagó a tiempo, 0 = no) |

### Unificación de valores nulos

Se empieza con la variable `tendencia_ingresos` ya que nos muestra 46 posibles valores cuando teniendo una logica solo serian 3 variables ya que la tendencia de los ingresos serian Creciennte, Decreciente o Estables 

In [19]:
df["tendencia_ingresos"].value_counts(dropna=False)

tendencia_ingresos
Creciente      5294
NaN            2932
Decreciente    1291
Estable        1188
0                 7
8315              6
1000000           4
9147              2
158042            1
3978              1
168750            1
-28589            1
-566272           1
24702             1
31837             1
122727            1
417087            1
9090              1
173031            1
-70715            1
-435177           1
-702927           1
-4105             1
54683             1
22832             1
209090            1
5697              1
-288              1
-164315           1
2029000           1
17181             1
15245             1
82657             1
52862             1
1817052           1
75761             1
146918            1
1123000           1
15090             1
4250635           1
22363             1
-101368           1
86286             1
65988             1
77975             1
-224714           1
10808             1
Name: count, dtype: int64

In [20]:
categorias_validas = ["Creciente", "Estable", "Decreciente"]

df["tendencia_ingresos"] = df["tendencia_ingresos"].where(
    df["tendencia_ingresos"].isin(categorias_validas)
)

df["tendencia_ingresos"].value_counts(dropna=False)

tendencia_ingresos
Creciente      5294
NaN            2990
Decreciente    1291
Estable        1188
Name: count, dtype: int64

**Decisión:**
 Se conservan únicamente las categorías Creciente, Decreciente y Estable de la variable `tendencia_ingresos`. Los demás valores se tratan como nulos (`NaN`), ya que no hay suficiente información para asignarlos a una de las tres categorías.

**Justificación:**
- Semánticamente, una tendencia de ingresos solo puede crecer, decrecer o mantenerse estable.
- Estas tres categorías concentran el 99,3 % de los registros no nulos.
- Los 58 registros restantes (0,7 %) son valores numéricos, incluso negativos, sin relación con una tendencia; probablemente se deben a un error de carga.

Se reemplazan por `NaN` en lugar de eliminar las filas, para no perder la información de las demás variables. Los nulos de esta columna pasan de 2.932 a 2.990.

In [21]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
tipo_credito,10763.0,5.411131,4.0,4.0,4.0,9.0,68.0,2.338279
fecha_prestamo,10763,2025-04-16 23:06:02.111121,2024-11-26 09:17:04,2025-01-20 17:33:07.500000,2025-03-27 16:23:12,2025-06-16 13:27:58,2026-04-26 18:43:52,NaN
capital_prestado,10763.0,2434315.001319,360000.0,1224831.0,1921920.0,3084840.0,41444152.8,1909642.758997
plazo_meses,10763.0,10.575583,2.0,6.0,10.0,12.0,90.0,6.632082
edad_cliente,10763.0,43.94862,19.0,33.0,42.0,53.0,123.0,15.060877
salario_cliente,10763.0,17216431.459909,0.0,2000000.0,3000000.0,4875808.0,22000000000.0,355476717.603482
total_otros_prestamos,10763.0,6238869.648518,0.0,500000.0,1000000.0,2000000.0,6787675263.0,118418316.941069
cuota_pactada,10763.0,243617.406671,23944.0,121041.5,182863.0,287833.5,3816752.0,210493.694608
puntaje,10763.0,91.170036,-38.00999,95.227787,95.227787,95.227787,95.227787,16.465441
puntaje_datacredito,10757.0,780.790834,-7.0,757.0,791.0,825.0,999.0,104.878031


In [22]:
# Edades imposibles mayores a 100 años
df.loc[df["edad_cliente"] > 100, "edad_cliente"] = np.nan

# Puntaje Datacrédito fuera del rango oficial de Colombia (150 a 950)
fuera_rango = (df["puntaje_datacredito"] < 150) | (df["puntaje_datacredito"] > 950)
df.loc[fuera_rango, "puntaje_datacredito"] = np.nan

# Salario en 0 = dato no registrado
df.loc[df["salario_cliente"] == 0, "salario_cliente"] = np.nan

# Verificación
df[["edad_cliente", "puntaje_datacredito", "salario_cliente"]].isnull().sum()

edad_cliente           150
puntaje_datacredito    159
salario_cliente         24
dtype: int64

**Valores imposibles o nulos disfrazados**

Al visualizar el mínimo y el máximo de las variables con `describe()`, se encontraron valores que no son válidos y se reemplazaron por `NaN`, según las siguientes reglas:

`edad_cliente` personas que tubieran una edad > 100 años (150 registros): son edades biológicamente imposibles (121 a 123 años). Además, las edades válidas llegan hasta 69 y luego saltan directamente a 121, lo que confirma que se trata de un error.

`puntaje_datacredito` fuera de 150 a 950 (153 registros): el puntaje de Datacrédito maneja una escala de 150 a 950 ([La FM](https://www.lafm.com.co/economia/cual-es-el-mejor-puntaje-de-datacredito)). Valores como 0, -7 o 999 no pueden ser generados por esa escala y probablemente representan "sin información".

`salario_cliente` = 0 (24 registros): a estos clientes se les otorgó un crédito, por lo que un salario de 0 se  puede interpreta como un dato no registrado o como un error al otorgar un credito por su ingreso real.



In [23]:
porcentaje_valor_frecuente = df.apply(
    lambda columna: columna.value_counts(normalize=True).iloc[0] * 100
).round(2)

porcentaje_valor_frecuente.sort_values(ascending=False).head(8)

saldo_mora_codeudor           99.97
saldo_mora                    99.48
Pago_atiempo                  95.25
puntaje                       87.40
creditos_sectorCooperativo    82.39
tipo_credito                  71.98
tendencia_ingresos            68.11
tipo_laboral                  62.75
dtype: float64

**Interpretación:**
 Estos datos nos muestran que proporción de registros comparte el valor más frecuente de cada variable. Valores superiores al 95-99 % indican variables casi constantes, que podrían no aportar información al modelo. 


In [24]:
df = df.drop(columns=["saldo_mora_codeudor"])
df.shape

(10763, 22)

### Eliminación de variables irrelevantes
`saldo_mora_codeudor` se elimina: el 99,97 % de los registros con dato tiene valor 0 solo 3 clientes tienen un valor distinto, lo que no permite aprender ningún patrón.

`saldo_mora` se conserva: aunque el sea 99,48 % que son 0, sin embargo es una variable que no puede favorecer para el modelo

`fecha_prestamo` se conserva:no se usará directamente como predictor, pero es útil para el análisis temporal y el monitoreo del modelo.